In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
from stl import mesh
import matplotlib.pyplot as plt
import re

In [ ]:
import sys
import os


from AutomationModule import * 

import AutomationModule as AM

In [ ]:
tidy3dAPI = os.environ["API_TIDY3D_KEY"]


In [ ]:
lambdas = np.array([7,2])

In [ ]:
folder_path = rf"../../Structures/N100000"
project_name = "20261124_LSU_T_L_Tests"
postprocess_results = []
runtime_ps = 70e-12
min_steps_per_lambda = 20
cuts=[0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1]
h5_bg = None
n_eff = 1

# Task-id files are stored in the repo-level data folder: /mnt/fast/Codes/tidy3d/data/<project_name>/...
# Every .txt keeps the "<ref id>\n<task id>" layout that AM.loadFromFile expects.
project_dir = Path("/mnt/fast/Codes/tidy3d/data") / project_name
project_dir.mkdir(parents=True, exist_ok=True)

# Single empty-domain reference for the whole project. It is valid for every cut because
# cut_cell=False keeps Lz fixed. Use a new project_name if lambdas, runtime or box size change.
ref_file = project_dir / "LSU_reference.txt"   # "\n<ref id>" -> load with get_ref=False

for direction in ["z"]:
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for filename in filenames:
            if not filename.endswith(".h5"):
                continue
            file_h5 = os.path.join(dirpath, filename)
            ff = float(re.search(r'ff_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            folder_desc = project_dir / f"n_{n_value:.2f}"
            folder_desc.mkdir(parents=True, exist_ok=True)

            for cut in cuts:
                sim_name = rf"LSU_{Path(filename).stem}_size_{cut}"
                run_file = folder_desc / f"{sim_name}.txt"   # "\n<task id>" (no own reference)
                if run_file.exists():
                    print(f"Exist! {sim_name}")
                    continue
                try:
                    structure_1 = AM.loadAndRunStructure(key = tidy3dAPI, file_path=file_h5
                                                    ,direction=direction, lambda_range=lambdas,
                                                    box_size=53.10,runtime_ps=runtime_ps,min_steps_per_lambda=min_steps_per_lambda,
                                                   scaling=1,shuoff_condtion=1e-9, verbose=True,
                                                   monitors=["flux"],sim_bg=n_eff**2, cut_cell=False,
                                                   freqs=400,
                                                   cut_condition=cut, source="planewave", absorbers=130, 
                                                   use_permittivity=False,sim_name=rf"{Path(filename).stem}_size_{cut}" + (rf"_bg_{h5_bg}" if h5_bg else ""),h5_bg=h5_bg,
                                                   )
                    sim = structure_1.sim

                    if not ref_file.exists():
                        sim0 = sim.copy(update={"structures": []})
                        id0 = web.upload(sim0, folder_name=project_name, task_name="LSU_reference", verbose=True)
                        web.start(task_id=id0)
                        ref_file.write_text("\n" + id0)
                        web.monitor(id0)

                    task_id = web.upload(sim, folder_name=project_name, task_name=sim_name, verbose=True)
                    web.start(task_id=task_id)
                    run_file.write_text("\n" + task_id)
                    web.monitor(task_id)
                except Exception as e:
                    print(f"Error processing {sim_name}: {e}")


In [ ]:
structure_1.plot_sim_layout()